# Developing the Agent logic

The whole point of the project is to find a way to use the differences of
performance which have proven to maybe be useful.

The steps to follow are according to the logical order of training with replay
buffer

0. Preparation for buffer (indexing and data-structure)
1. Feeding "forward pass" - including a simple actor
2. Sampling
3. training logic with buffer

In [1]:
import torch

import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)


from thesis.helper_scripts.general import extract_objs_from_sim_dir
from thesis.helper_scripts.hypothesis_visualization import simulation_counts_per_round

from berebasl.simulation.credit_data_simulation import CreditData, CreditDataSample


sim_objs = extract_objs_from_sim_dir(
    grid_path='../berebasl/data/simulations/mnar_grid_hyp_testing',
    sim_dir='bias_0_0_corr_0_2',
    device=torch.device('cpu')
)
credit_data = sim_objs["credit_data"]

In [3]:
from typing import Literal, Dict
from copy import deepcopy

class ReplayBuffer:
    def __init__(
            self,
            credit_data: CreditData,
            alternative_accepted: Dict[str, torch.Tensor],
            wished_acc_base: Literal["roc", "ks"]
        ):
        self.credit_data = deepcopy(credit_data)
        key_wanted_acc = "acc_based_" + wished_acc_base
        if key_wanted_acc in alternative_accepted:
            self.credit_data.change_acceptance_flag(alternative_accepted[key_wanted_acc])

buffer = ReplayBuffer(sim_objs["credit_data"], sim_objs["alternative_accepted"], wished_acc_base="roc")

In [15]:
counts, th_meaning = simulation_counts_per_round(sim_objs)

In [16]:
th_meaning

['through_the_door',
 'oracle_ks',
 'acc_based_roc',
 'oracle_roc',
 'acc_based_ks']